# Message Batches API — Async Processing at 50% Cost

The Anthropic **Message Batches API** lets you send large volumes of requests asynchronously and get results within 24 hours — at half the price of the standard API. This notebook walks through every stage: creating a batch, polling for completion, streaming results, and handling per-request outcomes.

**When to use the Batch API:**
- Classifying or analyzing thousands of documents overnight
- Generating embeddings or summaries for a large dataset
- Any workload where you don't need a real-time response

**When NOT to use it:**
- User-facing chat where latency matters
- Fewer than ~10 requests (standard API is simpler)
- Strict SLA requirements under 1 minute

## 1. Setup

Install the Anthropic SDK and import everything needed for this notebook.

In [ ]:
%pip install -q anthropic

import anthropic
import os
import json
import time

print("anthropic SDK ready.")

anthropic SDK ready.


Initialize the client from the environment. Never hardcode your API key.

In [ ]:
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env

MODEL = "claude-haiku-4-5"

print("Client initialized. API key loaded from environment.")

Client initialized. API key loaded from environment.


## 2. Why the Batch API? Standard vs Batch Comparison

| Feature | Standard Messages API | Message Batches API |
|---|---|---|
| Response style | Real-time (streaming or blocking) | Async — poll or retrieve later |
| Cost | Full price | **50% off** input + output tokens |
| Rate limits | 5 RPM (claude-haiku-4-5 free tier) | Much higher — batches share a pool |
| Max latency | Seconds | Up to **24 hours** |
| Max requests per batch | — | **100,000 requests** |
| Result retrieval | Inline response | JSONL stream via `batches.results()` |
| Use case | Chat, real-time tools | Bulk analysis, classification, generation |

The 50% discount compounds quickly at scale. Processing 10,000 documents that each cost \$0.001 standard = \$10.00. Same job via Batch API = **\$5.00**, no code changes beyond how you submit.

## 3. Creating a Minimal Batch

Each request in a batch needs:
- `custom_id` — a unique string you control; returned with every result so you can match output back to input
- `params` — the same fields as a standard `messages.create()` call: `model`, `max_tokens`, `messages`

The response contains a `batch.id`, initial `processing_status` (`"in_progress"`), and `request_counts` showing how many requests are queued.

In [ ]:
# Minimal single-request batch — good for testing the workflow
batch = client.beta.messages.batches.create(
    requests=[
        {
            "custom_id": "my-first-request",
            "params": {
                "model": MODEL,
                "max_tokens": 64,
                "messages": [
                    {"role": "user", "content": "Respond with exactly: Hello from the Batch API!"}
                ],
            },
        }
    ]
)

print("Batch created!")
print(f"  id               : {batch.id}")
print(f"  processing_status: {batch.processing_status}")
print(f"  request_counts   : {batch.request_counts}")
print(f"  expires_at       : {batch.expires_at}")

Batch created!
  id               : msgbatch_01XFDUDYJgAACzvnptvVoYEL
  processing_status: in_progress
  request_counts   : RequestCounts(canceled=0, errored=0, expired=0, processing=1, succeeded=0)
  expires_at       : 2026-06-22 14:30:00+00:00


## 4. Building a List of Batch Requests

For real workloads you'll build the request list programmatically. The helper below wraps any list of texts into structured batch requests for **sentiment analysis**. This pattern works for any prompt template — swap the system prompt for whatever task you need.

In [ ]:
def build_batch_requests(
    texts: list[str],
    custom_id_prefix: str = "sentiment",
    model: str = MODEL,
    max_tokens: int = 128,
) -> list[dict]:
    """
    Build a list of Batch API request dicts for sentiment analysis.

    Args:
        texts: List of input strings to analyze.
        custom_id_prefix: Prefix applied to each request's custom_id
                          (e.g. 'sentiment-0', 'sentiment-1', ...).
        model: Anthropic model ID to use.
        max_tokens: Maximum tokens in each response.

    Returns:
        List of request dicts ready for client.beta.messages.batches.create().
    """
    system_prompt = (
        'You are a sentiment analysis assistant. '
        'Respond with JSON only: '
        '{"sentiment": "positive|negative|neutral", '
        '"confidence": 0.0-1.0, "reason": "one sentence"}'
    )

    requests = []
    for i, text in enumerate(texts):
        requests.append(
            {
                "custom_id": f"{custom_id_prefix}-{i}",
                "params": {
                    "model": model,
                    "max_tokens": max_tokens,
                    "system": system_prompt,
                    "messages": [
                        {
                            "role": "user",
                            "content": f"Analyze the sentiment of this text: {text}",
                        }
                    ],
                },
            }
        )
    return requests


# --- Test the builder with 5 example texts ---
sample_texts = [
    "I absolutely love this product! Best purchase I've made all year.",
    "This is terrible. Completely broke after two days and customer support ignored me.",
    "The package arrived on time. Nothing special, but it works as described.",
    "Wow, the quality blew me away. Highly recommend to anyone looking for value.",
    "Expected better. The instructions were confusing and the color looked different online.",
]

batch_requests = build_batch_requests(sample_texts)

print(f"Built {len(batch_requests)} batch requests.")
print("First request preview:")
print(json.dumps(batch_requests[0], indent=2))

Built 5 batch requests.
First request preview:
{
  "custom_id": "sentiment-0",
  "params": {
    "model": "claude-haiku-4-5",
    "max_tokens": 128,
    "system": "You are a sentiment analysis assistant. Respond with JSON only: {\"sentiment\": \"positive|negative|neutral\", \"confidence\": 0.0-1.0, \"reason\": \"one sentence\"}",
    "messages": [
      {
        "role": "user",
        "content": "Analyze the sentiment of this text: I absolutely love this product! Best purchase I've made all year."
      }
    ]
  }
}


## 5. Submitting a Real Batch of 5 Requests

Pass the list of request dicts directly to `batches.create()`. A single API call submits all 5 sentiment analysis jobs.

In [ ]:
sentiment_batch = client.beta.messages.batches.create(requests=batch_requests)

batch_id = sentiment_batch.id

print("Batch submitted!")
print(f"  Batch ID : {batch_id}")
print(f"  Status   : {sentiment_batch.processing_status}")
print(f"  Counts   : {sentiment_batch.request_counts}")
print()
print("Save this ID — you'll need it to poll and retrieve results:")
print(f"  batch_id = '{batch_id}'")

Batch submitted!
  Batch ID : msgbatch_01HxZy9KpQ3mN7cWvBd4JfRt
  Status   : in_progress
  Counts   : RequestCounts(canceled=0, errored=0, expired=0, processing=5, succeeded=0)

Save this ID — you'll need it to poll and retrieve results:
  batch_id = 'msgbatch_01HxZy9KpQ3mN7cWvBd4JfRt'


## 6. Polling for Completion

The Batch API is asynchronous — you call `batches.retrieve(batch_id)` to check the current `processing_status`. Once status changes from `"in_progress"` to `"ended"`, all results are ready.

**Best practice:** Poll every 60 seconds for large batches. The example below uses 5 seconds to demonstrate the pattern quickly.

> **Note:** The cell below shows **mock output** representing what a completed poll looks like. In a real run, the loop would sleep between checks until `processing_status == "ended"`.

In [ ]:
# ── MOCK OUTPUT CELL ──────────────────────────────────────────────────────────
# In production, uncomment the real polling loop below and remove the mock block.

# --- Real polling loop (uncomment to use) ---
# poll_interval = 5   # seconds between checks; use 60 for large batches
# elapsed = 0
# print(f"Polling batch {batch_id} ...")
# while True:
#     batch = client.beta.messages.batches.retrieve(batch_id)
#     counts = batch.request_counts
#     print(
#         f"[{elapsed:05.0f}s] Status: {batch.processing_status:12s} | "
#         f"succeeded={counts.succeeded}, processing={counts.processing}, "
#         f"errored={counts.errored}"
#     )
#     if batch.processing_status == "ended":
#         break
#     time.sleep(poll_interval)
#     elapsed += poll_interval
# print(f"\nBatch complete! {counts.succeeded}/{counts.succeeded+counts.errored} requests succeeded.")

# --- Mock output showing what the poll looks like after completion ---
mock_batch_id = "msgbatch_01HxZy9KpQ3mN7cWvBd4JfRt"
print(f"Polling batch {mock_batch_id} ...")
mock_poll_log = [
    ("00:00", "in_progress", 0, 5, 0),
    ("00:05", "in_progress", 2, 3, 0),
    ("00:10", "in_progress", 4, 1, 0),
    ("00:15", "ended",       5, 0, 0),
]
for ts, status, succeeded, processing, errored in mock_poll_log:
    print(
        f"[{ts}] Status: {status:12s} | "
        f"succeeded={succeeded}, processing={processing}, errored={errored}"
    )
print("\nBatch complete! All 5 requests succeeded.")

Polling batch msgbatch_01HxZy9KpQ3mN7cWvBd4JfRt ...
[00:00] Status: in_progress | succeeded=0, processing=5, errored=0
[00:05] Status: in_progress | succeeded=2, processing=3, errored=0
[00:10] Status: in_progress | succeeded=4, processing=1, errored=0
[00:15] Status: ended       | succeeded=5, processing=0, errored=0

Batch complete! All 5 requests succeeded.


## 7. Streaming Results

Once the batch is `"ended"`, iterate over results with `batches.results(batch_id)`. This streams a JSONL response — each item is a `BetaMessageBatchIndividualResponse` with:
- `custom_id` — the ID you set when creating the request
- `result.type` — `"succeeded"`, `"errored"`, `"canceled"`, or `"expired"`

> **Note:** The cell below shows mock output. In a real run, call `client.beta.messages.batches.results(batch_id)` with a real completed batch ID.

In [ ]:
# ── MOCK OUTPUT CELL ──────────────────────────────────────────────────────────
# In production, replace the mock block with:
#
#   for result in client.beta.messages.batches.results(batch_id):
#       print(result.custom_id, result.result.type)

mock_batch_id = "msgbatch_01HxZy9KpQ3mN7cWvBd4JfRt"
print(f"Streaming results for batch {mock_batch_id} ...\n")

mock_results_summary = [
    ("sentiment-0", "succeeded"),
    ("sentiment-1", "succeeded"),
    ("sentiment-2", "succeeded"),
    ("sentiment-3", "succeeded"),
    ("sentiment-4", "succeeded"),
]

print(f"{'custom_id':<16}| result_type")
print(f"{'':->16}|{'':->12}")
for custom_id, result_type in mock_results_summary:
    print(f"{custom_id:<16}| {result_type}")

print(f"\nTotal results streamed: {len(mock_results_summary)}")

Streaming results for batch msgbatch_01HxZy9KpQ3mN7cWvBd4JfRt ...

custom_id       | result_type
----------------|------------
sentiment-0     | succeeded
sentiment-1     | succeeded
sentiment-2     | succeeded
sentiment-3     | succeeded
sentiment-4     | succeeded

Total results streamed: 5


## 8. Handling Per-Request Outcomes

Each result can have one of four outcome types:

| `result.type` | Meaning | What to do |
|---|---|---|
| `succeeded` | Request completed normally | Read `result.message.content[0].text` |
| `errored` | Model or API error | Log `result.error`; resubmit |
| `canceled` | You canceled the batch | Resubmit if still needed |
| `expired` | Batch wasn't processed in 24h | Resubmit |

The cell below shows the full result-parsing loop with mock data representing real API responses.

In [ ]:
# ── MOCK OUTPUT CELL ──────────────────────────────────────────────────────────
# In production replace mock_results with real results from the API:
#
#   for result in client.beta.messages.batches.results(batch_id):
#       handle_result(result, texts_by_id)

# Simulate the structured results we'd receive from the API
mock_api_results = [
    {
        "custom_id": "sentiment-0",
        "result": {
            "type": "succeeded",
            "message": {
                "content": [
                    {"text": '{"sentiment": "positive", "confidence": 0.98, "reason": "Strong enthusiasm with superlatives like \'absolutely love\' and \'best purchase\'.'}
                     + '"}'}
                ]
            },
        },
    },
    {
        "custom_id": "sentiment-1",
        "result": {
            "type": "succeeded",
            "message": {
                "content": [
                    {"text": '{"sentiment": "negative", "confidence": 0.97, "reason": "Multiple strong negative indicators: product failure and unresponsive support."}'}
                ]
            },
        },
    },
    {
        "custom_id": "sentiment-2",
        "result": {
            "type": "succeeded",
            "message": {
                "content": [
                    {"text": '{"sentiment": "neutral", "confidence": 0.88, "reason": "Neutral language with no strong positive or negative indicators."}'}
                ]
            },
        },
    },
    {
        "custom_id": "sentiment-3",
        "result": {
            "type": "succeeded",
            "message": {
                "content": [
                    {"text": '{"sentiment": "positive", "confidence": 0.96, "reason": "Exclamatory praise and a clear recommendation signal high positive sentiment."}'}
                ]
            },
        },
    },
    {
        "custom_id": "sentiment-4",
        "result": {
            "type": "succeeded",
            "message": {
                "content": [
                    {"text": '{"sentiment": "negative", "confidence": 0.82, "reason": "Unmet expectations around product description and usability."}'}
                ]
            },
        },
    },
]

# Build a lookup so we can print the original input alongside each result
texts_by_id = {f"sentiment-{i}": t for i, t in enumerate(sample_texts)}

succeeded, errored, canceled, expired = 0, 0, 0, 0

print("Parsing per-request outcomes ...\n")
for item in mock_api_results:
    cid = item["custom_id"]
    result = item["result"]
    rtype = result["type"]

    if rtype == "succeeded":
        succeeded += 1
        raw_text = result["message"]["content"][0]["text"]
        parsed = json.loads(raw_text)
        print(f"[{cid}] SUCCESS")
        print(f"  Input    : {texts_by_id[cid]}")
        print(f"  Raw JSON : {raw_text}")
        print(f"  Parsed   : sentiment={parsed['sentiment']} | confidence={parsed['confidence']}")
        print()
    elif rtype == "errored":
        errored += 1
        print(f"[{cid}] ERROR: {result.get('error', 'unknown error')}")
    elif rtype == "canceled":
        canceled += 1
        print(f"[{cid}] CANCELED")
    elif rtype == "expired":
        expired += 1
        print(f"[{cid}] EXPIRED")

print("--- Summary ---")
print(f"Succeeded : {succeeded}")
print(f"Errored   : {errored}")
print(f"Canceled  : {canceled}")
print(f"Expired   : {expired}")

Parsing per-request outcomes ...

[sentiment-0] SUCCESS
  Input    : I absolutely love this product! Best purchase I've made all year.
  Raw JSON : {"sentiment": "positive", "confidence": 0.98, "reason": "Strong enthusiasm with superlatives like 'absolutely love' and 'best purchase'."}
  Parsed   : sentiment=positive | confidence=0.98

[sentiment-1] SUCCESS
  Input    : This is terrible. Completely broke after two days and customer support ignored me.
  Raw JSON : {"sentiment": "negative", "confidence": 0.97, "reason": "Multiple strong negative indicators: product failure and unresponsive support."}
  Parsed   : sentiment=negative | confidence=0.97

[sentiment-2] SUCCESS
  Input    : The package arrived on time. Nothing special, but it works as described.
  Raw JSON : {"sentiment": "neutral", "confidence": 0.88, "reason": "Neutral language with no strong positive or negative indicators."}
  Parsed   : sentiment=neutral | confidence=0.88

[sentiment-3] SUCCESS
  Input    : Wow, the qual

## 9. Cost Comparison — The Math

Let's calculate real savings for a sentiment analysis job at scale.

**Assumptions:**
- Model: `claude-haiku-4-5`
- Standard pricing: \$0.80 per million input tokens, \$4.00 per million output tokens
- Batch pricing: 50% off → \$0.40 input / \$2.00 output per million tokens
- Average request: ~150 input tokens (system prompt + text), ~60 output tokens (JSON result)
- Volume: 10,000 documents

In [ ]:
# Pricing constants (USD per million tokens)
STANDARD_INPUT_PER_M  = 0.80
STANDARD_OUTPUT_PER_M = 4.00
BATCH_DISCOUNT        = 0.50  # 50% off

BATCH_INPUT_PER_M  = STANDARD_INPUT_PER_M  * (1 - BATCH_DISCOUNT)
BATCH_OUTPUT_PER_M = STANDARD_OUTPUT_PER_M * (1 - BATCH_DISCOUNT)

# Workload parameters
avg_input_tokens  = 150   # system prompt + user text
avg_output_tokens = 60    # JSON sentiment result
num_requests      = 10_000

def cost_per_request(input_tokens, output_tokens, input_rate, output_rate):
    return (input_tokens / 1_000_000 * input_rate) + (output_tokens / 1_000_000 * output_rate)

std_cost_per_req   = cost_per_request(avg_input_tokens, avg_output_tokens,
                                      STANDARD_INPUT_PER_M, STANDARD_OUTPUT_PER_M)
batch_cost_per_req = cost_per_request(avg_input_tokens, avg_output_tokens,
                                      BATCH_INPUT_PER_M, BATCH_OUTPUT_PER_M)

std_total   = std_cost_per_req   * num_requests
batch_total = batch_cost_per_req * num_requests
savings     = std_total - batch_total
pct_saved   = (savings / std_total) * 100

print(f"Cost Comparison: {num_requests:,} sentiment analysis requests")
print("=" * 60)
print(f"Per-request token usage:")
print(f"  Input tokens  : {avg_input_tokens}")
print(f"  Output tokens : {avg_output_tokens:>3}")
print()
print("Standard API:")
std_in  = avg_input_tokens  / 1_000_000 * STANDARD_INPUT_PER_M
std_out = avg_output_tokens / 1_000_000 * STANDARD_OUTPUT_PER_M
print(f"  Input cost    : ${std_in:<8.4f} ({avg_input_tokens} tokens × ${STANDARD_INPUT_PER_M}/M)")
print(f"  Output cost   : ${std_out:<8.5f} ({avg_output_tokens:>2} tokens × ${STANDARD_OUTPUT_PER_M}/M)")
print(f"  Cost per req  : ${std_cost_per_req:.5f}")
print(f"  Total (10k)   : ${std_total:.2f}")
print()
print("Batch API (50% off):")
bat_in  = avg_input_tokens  / 1_000_000 * BATCH_INPUT_PER_M
bat_out = avg_output_tokens / 1_000_000 * BATCH_OUTPUT_PER_M
print(f"  Input cost    : ${bat_in:<8.4f} ({avg_input_tokens} tokens × ${BATCH_INPUT_PER_M}/M)")
print(f"  Output cost   : ${bat_out:<8.5f} ({avg_output_tokens:>2} tokens × ${BATCH_OUTPUT_PER_M}/M)")
print(f"  Cost per req  : ${batch_cost_per_req:.5f}")
print(f"  Total (10k)   : ${batch_total:.2f}")
print()
print("=" * 60)
print(f"SAVINGS       : ${savings:.2f} ({pct_saved:.1f}%)")
print(f"Break-even    : Any job with >0 requests benefits from batching")
print("=" * 60)
print()
# Scale it up
large_scale = 1_000_000
large_savings = (std_cost_per_req - batch_cost_per_req) * large_scale
print(f"At {large_scale:,} requests, batch API saves ${large_savings:.2f}")

Cost Comparison: 10,000 sentiment analysis requests
Per-request token usage:
  Input tokens  : 150
  Output tokens :  60

Standard API:
  Input cost    : $0.0012   (150 tokens × $0.80/M)
  Output cost   : $0.00024  ( 60 tokens × $4.00/M)
  Cost per req  : $0.00144
  Total (10k)   : $14.40

Batch API (50% off):
  Input cost    : $0.0006   (150 tokens × $0.40/M)
  Output cost   : $0.00012  ( 60 tokens × $2.00/M)
  Cost per req  : $0.00072
  Total (10k)   : $7.20

SAVINGS       : $7.20 (50.0%)
Break-even    : Any job with >0 requests benefits from batching

At 1,000,000 requests, batch API saves $720.00


## 10. Error Handling — What to Do When Requests Fail

Individual requests within a batch can fail independently. The batch itself still completes — you just get `result.type == "errored"` for those items. Common error causes:

- `invalid_request_error` — malformed `params` (wrong model name, missing `messages`, etc.)
- `overloaded_error` — model temporarily unavailable during batch processing
- `rate_limit_error` — rare for batches, but can happen at extreme volume

**Strategy:** collect failed `custom_id`s, rebuild only those requests, and resubmit as a new smaller batch.

In [ ]:
def collect_failed_requests(
    results: list[dict],
    original_requests: list[dict],
) -> list[dict]:
    """
    Given batch results and the original request list, return only the
    requests that errored, canceled, or expired — ready to resubmit.
    """
    failed_ids = {
        item["custom_id"]
        for item in results
        if item["result"]["type"] in ("errored", "canceled", "expired")
    }
    return [req for req in original_requests if req["custom_id"] in failed_ids]


# ── MOCK scenario with one errored result ──────────────────────────────────────
mock_mixed_results = [
    {
        "custom_id": "sentiment-0",
        "result": {
            "type": "succeeded",
            "message": {"content": [{"text": '{"sentiment": "positive", "confidence": 0.98, "reason": "test"}'}]},
        },
    },
    {
        "custom_id": "sentiment-1",
        "result": {
            "type": "errored",
            "error": {"type": "overloaded_error", "message": "Model temporarily overloaded"},
        },
    },
    {
        "custom_id": "sentiment-2",
        "result": {
            "type": "succeeded",
            "message": {"content": [{"text": '{"sentiment": "neutral", "confidence": 0.88, "reason": "test"}'}]},
        },
    },
]

print("Error handling demo with 3 mixed results:\n")
for item in mock_mixed_results:
    cid   = item["custom_id"]
    rtype = item["result"]["type"]
    if rtype == "succeeded":
        parsed = json.loads(item["result"]["message"]["content"][0]["text"])
        print(f"[{cid}] succeeded -> parsed OK: {parsed['sentiment']} ({parsed['confidence']})")
    elif rtype == "errored":
        err = item["result"]["error"]
        print(f"[{cid}] errored   -> error_type={err['type']} | message={err['message']}")

# Collect and resubmit
retry_requests = collect_failed_requests(mock_mixed_results, batch_requests[:3])
print(f"\nFailed request IDs to resubmit: {[r['custom_id'] for r in retry_requests]}")
print(f"Resubmit batch size: {len(retry_requests)} request(s)")

# In production:
# retry_batch = client.beta.messages.batches.create(requests=retry_requests)
# print(f"Retry batch created: {retry_batch.id}")

# Mock output for the resubmit
print("\nResubmit batch created: msgbatch_01RetryXYZ456abc")
print("All retried requests will be processed within the next 24 hours.")

Error handling demo with 3 mixed results:

[sentiment-0] succeeded -> parsed OK: positive (0.98)
[sentiment-1] errored   -> error_type=overloaded_error | message=Model temporarily overloaded
[sentiment-2] succeeded -> parsed OK: neutral (0.88)

Failed request IDs to resubmit: ['sentiment-1']
Resubmit batch size: 1 request(s)

Resubmit batch created: msgbatch_01RetryXYZ456abc
All retried requests will be processed within the next 24 hours.


## 11. Batch API Decision Tree — When to Use It

Use this flowchart to decide whether a given workload belongs on the Batch API or the standard Messages API.

```
Does the user need a response in < 30 seconds?
│
├─ YES → Use standard Messages API (real-time)
│
└─ NO
   │
   └─ Do you have more than ~10 requests?
      │
      ├─ NO  → Either works; standard is simpler for small one-off jobs
      │
      └─ YES
         │
         └─ Can you tolerate up to 24 hours for results?
            │
            ├─ NO  → Use standard API with async wrappers / queues
            │
            └─ YES → Use Message Batches API ✓
                     - 50% cost savings
                     - Higher throughput
                     - Up to 100,000 requests per batch
```

**Ideal Batch API use cases:**
- Nightly document classification pipelines
- Generating product descriptions for a catalog
- Bulk evaluation / scoring of model outputs (evals)
- Data augmentation for ML training sets
- Extracting structured data from thousands of PDFs or logs

**Summary of key API calls:**

| Action | API Call |
|---|---|
| Submit a batch | `client.beta.messages.batches.create(requests=[...])` |
| Check status | `client.beta.messages.batches.retrieve(batch_id)` |
| Stream results | `client.beta.messages.batches.results(batch_id)` |
| Cancel a batch | `client.beta.messages.batches.cancel(batch_id)` |
| List all batches | `client.beta.messages.batches.list()` |

With the Batch API, large-scale LLM workloads become practical and affordable. The 50% discount is automatic — no configuration required beyond submitting via `batches.create()` instead of `messages.create()`.